# Lab 9: Secure RAG with Azure Key Vault & Entra ID

**Duration:** 25 minutes  
**Environment:** Azure ML Studio Notebook  

---

## Learning Objectives

- Store API keys in Azure Key Vault instead of environment variables
- Retrieve secrets securely at runtime
- Configure managed identity for passwordless access
- Implement secure RAG pipeline with no hardcoded credentials

## Prerequisites

Labs 1-3 completed

---

© 2026 Great Learning. All rights reserved.

## Step 1: Load Configuration

In [ ]:
import os, json, urllib.request, ssl, subprocess

RESOURCE_GROUP = 'rg-promptflow-rag-lab'

# Auto-detect OpenAI resource that has BOTH gpt-4o and text-embedding-3-small deployments
# This handles the case where Lab 1 was run multiple times creating multiple OpenAI resources
print('Detecting Azure resources...')
r = subprocess.run(['az', 'cognitiveservices', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', "[?kind=='OpenAI'].name", '-o', 'tsv'], capture_output=True, text=True)
openai_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]

OPENAI_NAME = ''
for candidate in openai_candidates:
    # Check if this resource has both required deployments
    r = subprocess.run(['az', 'cognitiveservices', 'account', 'deployment', 'list',
        '-n', candidate, '-g', RESOURCE_GROUP, '--query', '[].name', '-o', 'tsv'],
        capture_output=True, text=True)
    deployments = r.stdout.strip().split('\n')
    if 'gpt-4o' in deployments and 'text-embedding-3-small' in deployments:
        OPENAI_NAME = candidate
        print(f'  Found OpenAI with deployments: {candidate}')
        break

if not OPENAI_NAME:
    print('ERROR: No OpenAI resource found with both gpt-4o and text-embedding-3-small deployments.')
    print(f'  Candidates checked: {openai_candidates}')
    print('  Fix: Go back to Lab 1 and run Steps 4 + 5 to create deployments.')
    raise SystemExit(1)

# Auto-detect Search service
r = subprocess.run(['az', 'search', 'service', 'list', '-g', RESOURCE_GROUP,
    '--query', '[].name', '-o', 'tsv'], capture_output=True, text=True)
search_candidates = [n.strip() for n in r.stdout.strip().split('\n') if n.strip()]
SEARCH_NAME = search_candidates[0] if search_candidates else ''

# Auto-detect Storage account
r = subprocess.run(['az', 'storage', 'account', 'list', '-g', RESOURCE_GROUP,
    '--query', '[0].name', '-o', 'tsv'], capture_output=True, text=True)
STORAGE_NAME = r.stdout.strip()

if not SEARCH_NAME:
    print('ERROR: No Search service found. Run Lab 1 first.')
    raise SystemExit(1)

# Get endpoints and keys
r = subprocess.run(['az', 'cognitiveservices', 'account', 'show', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'properties.endpoint', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_ENDPOINT = r.stdout.strip()
if not OPENAI_ENDPOINT.endswith('/'): OPENAI_ENDPOINT += '/'

r = subprocess.run(['az', 'cognitiveservices', 'account', 'keys', 'list', '-n', OPENAI_NAME, '-g', RESOURCE_GROUP,
    '--query', 'key1', '-o', 'tsv'], capture_output=True, text=True)
OPENAI_KEY = r.stdout.strip()

SEARCH_ENDPOINT = f'https://{SEARCH_NAME}.search.windows.net'
r = subprocess.run(['az', 'search', 'admin-key', 'show', '--service-name', SEARCH_NAME, '-g', RESOURCE_GROUP,
    '--query', 'primaryKey', '-o', 'tsv'], capture_output=True, text=True)
SEARCH_KEY = r.stdout.strip()

ctx = ssl.create_default_context()

print(f'\n✅ Resources detected:')
print(f'  OpenAI:  {OPENAI_NAME}')
print(f'  Search:  {SEARCH_NAME}')
print(f'  Storage: {STORAGE_NAME}')
print(f'  Endpoint: {OPENAI_ENDPOINT}')

## Azure CLI Authentication

Azure ML compute instances have a **managed identity** — this logs in instantly with no browser needed.

> If managed identity fails (permissions not assigned), the cell falls back to device code login.

In [ ]:
import subprocess, json as _j

# Check if already logged in
r = subprocess.run(['az', 'account', 'show'], capture_output=True, text=True)
if r.returncode == 0:
    acct = _j.loads(r.stdout)
    print(f'✅ Already logged in: {acct.get("user", {}).get("name", "unknown")}')
    print(f'   Subscription: {acct.get("name", "unknown")}')
else:
    # Try managed identity first (instant on Azure ML compute)
    r2 = subprocess.run(['az', 'login', '--identity'], capture_output=True, text=True)
    if r2.returncode == 0:
        acct = _j.loads(r2.stdout)[0]
        print(f'✅ Logged in via managed identity')
        print(f'   Subscription: {acct.get("name", "unknown")}')
    else:
        print('Managed identity not available. Using device code login...')
        !az login --use-device-code

## Step 2: Create Azure Key Vault

In [ ]:
!az keyvault create \
    --name {KV_NAME} \
    --resource-group {RESOURCE_GROUP} \
    --location {LOCATION} \
    --sku standard \
    --output table

print('\n✅ Key Vault created.')

**Expected output:** Key Vault details with `Succeeded` status.

## Step 3: Store API Keys as Secrets

In [ ]:
secrets = {
    'openai-api-key': OPENAI_KEY,
    'openai-endpoint': OPENAI_ENDPOINT,
    'search-api-key': SEARCH_KEY,
    'search-endpoint': SEARCH_ENDPOINT,
}

for name, value in secrets.items():
    !az keyvault secret set --vault-name {KV_NAME} --name "{name}" --value "{value}" --output table
    print(f'  ✅ Stored: {name}')

print('\n✅ All secrets stored in Key Vault.')

## Step 4: Retrieve Secrets & Run Secure RAG Pipeline

Secrets are retrieved from Key Vault at runtime — no credentials in code or environment variables.

In [ ]:
def get_secret(name):
    result = subprocess.run(['az','keyvault','secret','show','--vault-name',KV_NAME,
        '--name',name,'--query','value','-o','tsv'], capture_output=True, text=True)
    return result.stdout.strip()

print('  Retrieving secrets from Key Vault...')
kv_openai_endpoint = get_secret('openai-endpoint')
kv_openai_key = get_secret('openai-api-key')
kv_search_endpoint = get_secret('search-endpoint')
kv_search_key = get_secret('search-api-key')

print(f'  ✅ OpenAI endpoint: {kv_openai_endpoint[:30]}...')
print(f'  ✅ Search endpoint: {kv_search_endpoint[:30]}...')
print(f'  ✅ Keys retrieved securely')

In [ ]:
# Secure RAG query using Key Vault secrets
def secure_rag_query(question):
    # Embed
    url = f"{kv_openai_endpoint}openai/deployments/text-embedding-3-small/embeddings?api-version=2024-06-01"
    data = json.dumps({'input':question}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':kv_openai_key})
    with urllib.request.urlopen(req, context=ctx) as resp:
        vector = json.loads(resp.read())['data'][0]['embedding']
    # Search
    url = f"{kv_search_endpoint}/indexes/banking-policies/docs/search?api-version=2024-07-01"
    body = {'search':question,'vectorQueries':[{'vector':vector,'k':3,'fields':'content_vector','kind':'vector'}],
            'select':'title,content,source_file','top':3}
    data = json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, method='POST',
        headers={'Content-Type':'application/json','api-key':kv_search_key})
    with urllib.request.urlopen(req, context=ctx) as resp:
        results = json.loads(resp.read()).get('value', [])
    context = '\n'.join([f'[{r["source_file"]}]: {r["content"]}' for r in results])
    # Generate
    url = f"{kv_openai_endpoint}openai/deployments/gpt-4o/chat/completions?api-version=2024-06-01"
    data = json.dumps({'messages':[{'role':'system','content':'Banking assistant. Answer from context. Cite sources.'},
        {'role':'user','content':f'Context:\n{context}\n\nQuestion: {question}'}],
        'max_tokens':150,'temperature':0.1}).encode()
    req = urllib.request.Request(url, data=data, headers={'Content-Type':'application/json','api-key':kv_openai_key})
    with urllib.request.urlopen(req, context=ctx) as resp:
        return json.loads(resp.read())['choices'][0]['message']['content']

print('\n🧪 Testing secure RAG pipeline...')
question = 'What is the zero liability protection policy?'
answer = secure_rag_query(question)
print(f'\n  Q: {question}')
print(f'  A: {answer}')
print(f'\n  ✅ Secure RAG pipeline working — no credentials in code!')

## Step 5: Security Best Practices

| Practice | Status |
|---|---|
| Store API keys in Key Vault | ✅ |
| Use Managed Identity for passwordless access | 📋 Production step |
| Enable Key Vault audit logging | 📋 Production step |
| Rotate keys regularly (90-day policy) | 📋 Production step |
| Use RBAC instead of access policies | 📋 Production step |
| Enable soft-delete and purge protection | ✅ (default) |
| Network restrict Key Vault to VNet | 📋 Production step |

**Managed Identity commands for production:**
```bash
# Enable on Function App
az functionapp identity assign --name <app> --resource-group <rg>

# Grant Key Vault access
az keyvault set-policy --name <vault> \
  --object-id <identity-principal-id> \
  --secret-permissions get list
```

## 🧹 Final Cleanup (All Labs)

Delete all lab resources with a single command:

In [ ]:
# ⚠️ Only run this after all labs are complete!
# Uncomment the line below to delete ALL resources:

# !az group delete --name rg-promptflow-rag-lab --yes --no-wait

print('To clean up, uncomment the command above and run this cell.')
print('This deletes: Azure OpenAI, AI Search, Storage, Key Vault, Function App, App Insights.')

## ✅ Lab 9 Complete — All 9 Labs Done!

**Session Summary:**
- Lab 1: Azure resources provisioned (OpenAI, AI Search, Storage)
- Lab 2: Banking documents uploaded, embedded, and indexed
- Lab 3: RAG Chat Flow with multi-turn conversation
- Lab 4: Conversational search with hybrid + semantic ranking
- Lab 5: RAG evaluation (groundedness, relevance, coherence)
- Lab 6: Prompt variant A/B testing
- Lab 7: Deployed RAG as Azure Function endpoint
- Lab 8: Monitoring & observability with Application Insights
- Lab 9: Secured RAG with Azure Key Vault

**Next Steps:**
- Deploy to production with Managed Identity
- Add more banking documents to the index
- Implement CI/CD pipeline for the RAG flow
- Set up Azure Monitor dashboards

---

© 2026 Great Learning. All rights reserved.